## Compare:
#### 1 step displacement driven PC 
### VS 
#### 1 step displacement driven aire
### VS
#### 2 step force driven aire
### VS
#### multi step force driven PC
---
 - All basically identical 

In [92]:
from pathlib import Path 
from phd_helpers.AbaqusPostprocessing import inp2pv, get_field_path, get_field_df, add_field_to_mesh, get_history_path
import numpy as np
import pandas as pd

In [81]:
def get_mesh_data(inp_file, step, frame, field_metrics, history_metrics):
    """returns dictionary of tpm and mc1 meshes with mesh data in cell/point_data and history/summary data in field_data"""
    csv_dir = inp_file.parent / 'resultCSVs' 

    # get meshes
    meshes = inp2pv(inp_file)
    for bone, mesh in meshes.items():
        instance = f"{bone.upper()}_INST"
        
        # Field data
        for metric in field_metrics:
            field_path = get_field_path(csv_dir, metric, step, frame, instance)
            field_df = get_field_df(field_path)
            add_field_to_mesh(mesh, field_df)

        # History data
        history_data = pd.read_csv(get_history_path(csv_dir, step))
        # F
        RF_data = history_data[history_data['historyOutputKey']=='RF1']
        RF = np.abs(RF_data['value'].iloc[frame])
        # A
        CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']
        CA = CAREA_data['value'].iloc[frame]

        mesh.field_data['RF'] = RF
        mesh.field_data['CA'] = CA

        #Summary data
        mesh.field_data['P_max'] = mesh['CPRESS'].max()
        mesh.field_data['P_avg'] = np.mean(mesh['CPRESS'][mesh['CPRESS']>0])
        mesh.field_data['loc_Pmax'] = np.array(mesh.points[np.argmax(mesh['CPRESS'])])

    return meshes

def print_results_summary(meshes, dcs=3):
    for bone, mesh in meshes.items():
        print(f'\n--------- {bone} ---------\n')
        print(f'  Reaction force: {mesh.field_data['RF'][0]:.{dcs}f} N')
        print(f'    Contact area: {mesh.field_data['CA'][0]:.{dcs}f} mm^2')
        print(f'    Max pressure: {mesh.field_data['P_max'][0]:.{dcs}f} MPa')
        print(f'Average pressure: {mesh.field_data['P_avg'][0]:.{dcs}f} MPa')
        print(f'Max pressure loc: {str(tuple([round(float(x), dcs) for x in mesh.field_data['loc_Pmax']]))}')

def get_last_step_id(inp_file):
    history_files = list(inp_file.parent.glob('resultCSVs/history*.csv'))
    last_step_id = np.unique([int(x.with_suffix('').name.replace('history_step-', '')) for x in history_files]).max()
    return last_step_id

# Load data

In [60]:
field_metrics = ["CPRESS", "U"]
history_metrics = ['CAREA', 'RF']
frame = -1

In [88]:
fea_path = Path('../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy')
pc_file = list(fea_path.glob('study1_*5/inpFiles/**/*.inp'))[0]
aire_file = list(fea_path.glob('study1_*5/aire/output/**/*.inp'))[0]
aireF_file = list(fea_path.glob('study1_*5_F/aire/output/**/*.inp'))[0]
pcF_file = list(fea_path.glob('study3_*Fsteps/inpFiles/**/*.inp'))[0]

In [89]:
# pc results
pc_meshes = get_mesh_data(pc_file, get_last_step_id(pc_file), frame, field_metrics, history_metrics)
aire_meshes = get_mesh_data(aire_file, get_last_step_id(aire_file), frame, field_metrics, history_metrics)
aireF_meshes = get_mesh_data(aireF_file, get_last_step_id(aireF_file), frame, field_metrics, history_metrics)
pcF_meshes = get_mesh_data(pcF_file, get_last_step_id(pcF_file), frame, field_metrics, history_metrics)

In [82]:
print_results_summary(pc_meshes)


--------- tpm ---------

  Reaction force: 150.032 N
    Contact area: 68.692 mm^2
    Max pressure: 5.938 MPa
Average pressure: 1.590 MPa
Max pressure loc: (-19.872, 0.172, 3.977)

--------- mc1 ---------

  Reaction force: 150.032 N
    Contact area: 68.692 mm^2
    Max pressure: 5.941 MPa
Average pressure: 1.829 MPa
Max pressure loc: (-19.837, 0.148, 3.91)


In [83]:
print_results_summary(aire_meshes)


--------- tpm ---------

  Reaction force: 150.032 N
    Contact area: 68.692 mm^2
    Max pressure: 5.938 MPa
Average pressure: 1.590 MPa
Max pressure loc: (-19.872, 0.172, 3.977)

--------- mc1 ---------

  Reaction force: 150.032 N
    Contact area: 68.692 mm^2
    Max pressure: 5.941 MPa
Average pressure: 1.829 MPa
Max pressure loc: (-19.837, 0.148, 3.91)


In [84]:
print_results_summary(aireF_meshes) # Force controlled to 150 N
# RP_mc1 RF1 is 0 cos no longer contrained for force steps
# Need to output RP_tpm RF1
# - have altered runAbaqus code to include this


--------- tpm ---------

  Reaction force: 0.000 N
    Contact area: 68.689 mm^2
    Max pressure: 5.936 MPa
Average pressure: 1.590 MPa
Max pressure loc: (-19.872, 0.172, 3.977)

--------- mc1 ---------

  Reaction force: 0.000 N
    Contact area: 68.689 mm^2
    Max pressure: 5.939 MPa
Average pressure: 1.828 MPa
Max pressure loc: (-19.837, 0.148, 3.91)


In [91]:
# THIS ONE USED DIFFERENT MATERIAL MODELS #

print_results_summary(pcF_meshes) # Force controlled to 150 N
# RP_mc1 RF1 is 0 cos no longer contrained for force steps
# Need to output RP_tpm RF1
# - have altered runAbaqus code to include this


--------- tpm ---------

  Reaction force: 0.000 N
    Contact area: 72.723 mm^2
    Max pressure: 6.555 MPa
Average pressure: 1.604 MPa
Max pressure loc: (-19.849, -0.063, 4.213)

--------- mc1 ---------

  Reaction force: 0.000 N
    Contact area: 72.723 mm^2
    Max pressure: 6.599 MPa
Average pressure: 1.885 MPa
Max pressure loc: (-19.818, -0.056, 4.218)
